# Deep Convolutional Q-Learning for Pac-Man

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
# !pip install gymnasium
# !pip install "gymnasium[atari, accept-rom-license]"
# !pip install ale-py
# !apt-get install -y swig
# !pip install gymnasium[box2d]

In [ ]:
# pip install torchvision

### Importing the libraries

In [ ]:
# If PyTorch is missing, install the CUDA-enabled build selected on pytorch.org.
# In a notebook, use %pip (not plain "pip install"). The exact CUDA command
# depends on the NVIDIA driver and GPU, so it must be chosen for this machine.
# Example for a CPU-only fallback: %pip install torch

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

# Select the GPU when PyTorch can access CUDA; otherwise keep the notebook usable on CPU.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    # Fixed-size 128x128 frames benefit from cuDNN selecting an optimized convolution kernel.
    torch.backends.cudnn.benchmark = True
    print(f"GPU enabled: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}")
else:
    print("CUDA GPU unavailable: training will use the CPU.")

GPU enabled: Tesla T4 | CUDA 13.0


In [ ]:
# import os
# # Forza il riavvio immediato della sessione (kernel) di Colab
# os.kill(os.getpid(), 9)

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
class Network(nn.Module):

  def __init__(self, action_size, seed=42):
    # Eyes of the CNN

    # Initializes the base nn.Module class so that all PyTorch layers
    # are properly registered.
    super(Network, self).__init__()

    # Sets the PyTorch random seed so results are reproducible.
    self.seed = torch.manual_seed(seed)

    # 1st convolution: takes RGB images (3 channels) and extracts
    # 32 feature maps.
    # An 8x8 kernel with a stride of 4 significantly reduces
    # the image width and height.
    self.conv1 = nn.Conv2d(3, 32, kernel_size=8, stride=4)

    # Normalizes the 32 feature maps produced by conv1
    # to make training more stable.
    # Batch Normalization keeps the activations of each channel
    # in a more controlled distribution.
    self.bn1 = nn.BatchNorm2d(32)

    # 2nd convolution: combines the previous 32 feature maps
    # into 64 more abstract features.
    # Gradually increases the depth (# of channels) while
    # decreasing the spatial dimensions.
    self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)

    # Batch Normalization applied to the 64 feature maps from conv2.
    self.bn2 = nn.BatchNorm2d(64)

    # 3rd convolution: keeps 64 channels and looks for more local
    # details using a 3x3 kernel.
    self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

    # Batch Normalization applied to the 64 feature maps from conv3.
    self.bn3 = nn.BatchNorm2d(64)

    # 4th convolution: increases the number of feature maps
    # from 64 to 128, creating richer visual representations.
    self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1)

    # Batch Normalization applied to the 128 feature maps from conv4.
    self.bn4 = nn.BatchNorm2d(128)

    # Brain of the CNN

    # After the convolutional layers, the output consists of
    # 128 feature maps of size 10x10.
    # These 12,800 values are flattened and connected to 512 neurons.
    # This input size depends on the resolution of the original image.
    self.fc1 = nn.Linear(10 * 10 * 128, 512)

    # Reduces the representation from 512 to 256 neurons.
    self.fc2 = nn.Linear(512, 256)

    # Produces one Q-value for each possible action.
    # These are not probabilities, but estimates of expected return.
    self.fc3 = nn.Linear(256, action_size)

  # propagate the signal from the images to the first convolutional layer of the eyes
  def forward(self, state):
    x = F.relu(self.bn1(self.conv1(state)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))

    # Flattens the convolutional output while keeping
    # the batch dimension unchanged.
    x = x.view(x.size(0), -1)

    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))

    return self.fc3(x)

<img src="images/dcqn_architecture.png" width="1000">

# Part 2 - Training the AI

### Setting up the environment

In [ ]:
!pip install "gymnasium[atari, accept-rom-license]"

In [ ]:
import gymnasium as gym
import ale_py
# Create the MsPacman environment with deterministic behavior and a reduced action space so as to decrease environment complexity
# Deterministic -> a small part of actions (monsters) will be deterministic
# Full action space -> ensure the agent uses a simplified set of actions

env = gym.make('MsPacmanNoFrameskip-v0', full_action_space=False)
state_shape=env.observation_space.shape
state_size=env.observation_space.shape[0]
number_actions = env.action_space.n
print(f"State shape: {state_shape}, State size: {state_size}, Number of actions: {number_actions}")

State shape: (210, 160, 3), State size: 210, Number of actions: 9


/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:511: DeprecationWarning: WARN: The environment MsPacmanNoFrameskip-v0 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


In [ ]:
learning_rate = 5e-4
minibatch_size = 64
discount_factor = 0.99

NB: now the input states are images of a big size, so we cannot store 100.000 experiences containing all these images in the memory of our AI. We' ll make a simpler implementation of experience replay and we will preprocess the frames so as to convert the input images into pytorch tensors.


### Preprocessing the frames

In [ ]:
pip install Pillow

In [ ]:
from PIL import Image
from torchvision import transforms

def preprocess_frame(frame):
  frame = Image.fromarray(frame)
  preprocess = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])
  return preprocess(frame).unsqueeze(0)

In [ ]:
class Agent():

  def __init__(self, action_size):
    # Reuse the device selected above: CUDA GPU when available, otherwise CPU.
    # Models and every training batch are moved to this same device below.
    self.device = device

    # Store the size of the action space as an object variable.
    # For LunarLander, this is 4.
    self.action_size = action_size

    # local_qnetwork = the network that is directly trained and updated by gradient descent.
    # It has the same architecture as the Network class we defined before.
    self.local_qnetwork = Network(action_size).to(self.device)

    # target_qnetwork = a second Q-network with the same architecture.
    # It is used to compute more stable target values during learning.
    self.target_qnetwork = Network(action_size).to(self.device)

    # Optimizer used to update the weights of the local Q-network.
    # Adam takes the parameters (weights) of local_qnetwork and updates them
    # using the learning_rate defined earlier.
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr = learning_rate)

    self.memory=deque(maxlen = 10000)

  def step(self, state, action, reward, next_state, done):
    state = preprocess_frame(state)
    next_state = preprocess_frame(next_state)
    self.memory.append((state, next_state, action, reward, done))
    if len(self.memory) >= minibatch_size:
      experiences = random.sample(self.memory, k = minibatch_size)
      self.learn(experiences, discount_factor)

  def act(self, state, epsilon = 0.):
    # Convert the NumPy state into a float torch.Tensor, add the batch dimension
    # at index 0 with unsqueeze(0), and move it to the selected device (CPU or GPU).
    state = preprocess_frame(state).to(self.device)
    # Switch the local Q-network to evaluation mode before making predictions.
    self.local_qnetwork.eval()
    # Disable gradient computation because here we are only doing inference.
    with torch.no_grad():
      # Forward-pass the state through the local Q-network to get the Q-values
      # associated with each possible action.
      action_values = self.local_qnetwork(state)
    # Switch back to training mode after inference.
    self.local_qnetwork.train()
    # Epsilon-greedy policy:
    # if the random number is greater than epsilon, exploit by choosing the
    # action with the highest predicted Q-value.
    if random.random() > epsilon:
      return np.argmax(action_values.cpu().data.numpy())
    else:
      # Otherwise, explore by choosing a random action among all possible actions.
      return random.choice(np.arange(self.action_size))

  def learn(self, experiences, discount_factor):
    # Unpack the sampled experiences into their different elements:
    # current states, next states, actions taken, rewards received, and done flags.
    states, next_states, actions, rewards,   dones = zip(*experiences)
    states = torch.cat(states, dim=0).float().to(self.device)
    actions = torch.as_tensor(actions, dtype=torch.long, device=self.device).view(-1, 1)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=self.device).view(-1, 1)
    next_states = torch.cat(next_states, dim=0).float().to(self.device)
    dones = torch.as_tensor(dones, dtype=torch.float32, device=self.device).view(-1, 1)
    # Get the maximum predicted Q-values for the next states from the target network.
    # detach() removes this tensor from the computation graph, max(1)[0] keeps only
    # the maximum values along the action dimension, and unsqueeze(1) restores
    # the expected batch shape.
    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
    # Compute the Q-targets for the current states using the Bellman target:
    # reward + gamma * max_next_q * (1 - done).
    q_targets = rewards + discount_factor * next_q_targets * (1 - dones)
    # Get the expected Q-values from the local Q-network for the actions that
    # were actually taken in the sampled experiences.
    q_expected = self.local_qnetwork(states).gather(1, actions)
    # Compute the mean squared error loss between expected Q-values and target Q-values.
    loss = F.mse_loss(q_expected, q_targets)
    # Reset the optimizer gradients before backpropagation.
    self.optimizer.zero_grad()
    # Backpropagate the loss to compute gradients with respect to the model parameters.
    loss.backward()
    # Perform one optimization step to update the local network parameters.
    self.optimizer.step()

# Initializing the DCQN agent

In [ ]:
agent = Agent(number_actions)

In [ ]:
checkpoint_path = 'checkpoint.pth'

# Training the DCQN agent

In [16]:
# Maximum number of complete training episodes.
# One episode is one full attempt of the lander:
# the environment is reset, the agent acts step by step, and the episode ends
# when the lander lands, crashes, or reaches the time-step limit.
number_episodes = 2000

# Maximum number of time steps allowed inside one episode.
# At each time step, the agent observes a state, chooses an action,
# receives a reward, and moves to the next state.
# If the episode ends earlier, the inner loop stops before reaching this limit.
maximum_number_timesteps_per_episode = 10000

# Initial value of epsilon for the epsilon-greedy policy.
# At 1.0, the agent explores completely: it chooses random actions
# instead of relying on the Q-values predicted by the local network.
epsilon_starting_value  = 1.0

# Minimum value epsilon is allowed to reach.
# Even after many episodes, the agent keeps a small probability of exploration
# instead of becoming 100% greedy.
epsilon_ending_value  = 0.01

# Multiplicative decay applied to epsilon after each episode.
# This gradually reduces exploration over time:
# high exploration at the beginning, more exploitation later.
epsilon_decay_value  = 0.995

# Current value of epsilon used by agent.act(...).
# It starts from epsilon_starting_value and is updated after each episode.
epsilon = epsilon_starting_value

# Stores the scores from the most recent 100 episodes.
# The average of these scores is used to monitor training progress
# and to decide when the environment is considered solved.
scores_on_100_episodes = deque(maxlen = 100)

if os.path.exists(checkpoint_path):
  agent.local_qnetwork.load_state_dict(torch.load(checkpoint_path, map_location=agent.device))
  agent.local_qnetwork.eval()
  print(f'Loaded trained model from {os.path.abspath(checkpoint_path)}')
else:
  for episode in range(1, number_episodes + 1):
    state, _ = env.reset()
    score = 0
    for t in range(maximum_number_timesteps_per_episode):
      action = agent.act(state, epsilon)
      next_state, reward, terminated, truncated, _ = env.step(action)
      done = terminated or truncated
      agent.step(state, action, reward, next_state, done)
      state = next_state
      score += reward
      if done:
        break
    scores_on_100_episodes.append(score)
    epsilon = max(epsilon_ending_value, epsilon_decay_value * epsilon)
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)), end = "")
    if episode % 100 == 0:
      print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)))
    if np.mean(scores_on_100_episodes) >= 500.0:
      print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode - 100, np.mean(scores_on_100_episodes)))
      torch.save(agent.local_qnetwork.state_dict(), checkpoint_path)
      break


Episode 100	Average Score: 340.00
Episode 200	Average Score: 434.70
Episode 300	Average Score: 464.60
Episode 316	Average Score: 502.10
Environment solved in 216 episodes!	Average Score: 502.10
